In [15]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config

# Configure S3 client to use unsigned requests
s3 = boto3.client('s3', region_name='us-east-1', config=Config(signature_version=UNSIGNED))

bucket = 'broad-references'
prefix = 'hg38'

paginator = s3.get_paginator('list_objects_v2')
for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get('Contents', []):
        print(obj['Key'])



hg38/v0/1000G.phase3.integrated.sites_only.no_MATCHED_REV.hg38.vcf
hg38/v0/1000G.phase3.integrated.sites_only.no_MATCHED_REV.hg38.vcf.idx
hg38/v0/1000G_omni2.5.hg38.vcf.gz
hg38/v0/1000G_omni2.5.hg38.vcf.gz.tbi
hg38/v0/1000G_phase1.snps.high_confidence.hg38.vcf.gz
hg38/v0/1000G_phase1.snps.high_confidence.hg38.vcf.gz.tbi
hg38/v0/1000G_phase3_v4_20130502.sites.hg38.vcf
hg38/v0/1000G_phase3_v4_20130502.sites.hg38.vcf.idx
hg38/v0/Axiom_Exome_Plus.genotypes.all_populations.poly.hg38.vcf.gz
hg38/v0/Axiom_Exome_Plus.genotypes.all_populations.poly.hg38.vcf.gz.tbi
hg38/v0/CrossSpeciesContamination/ContaminantNormalizationFactors.txt
hg38/v0/CrossSpeciesContamination/CrossSpeciesContaminant/meats.dict
hg38/v0/CrossSpeciesContamination/CrossSpeciesContaminant/meats.fa
hg38/v0/CrossSpeciesContamination/CrossSpeciesContaminant/meats.fa.fai
hg38/v0/CrossSpeciesContamination/CrossSpeciesContaminant/meats.fa.img
hg38/v0/CrossSpeciesContamination/CrossSpeciesContaminant/meats.min2k.db
hg38/v0/CrossSpec

In [22]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config

s3 = boto3.client(
    "s3",
    region_name="us-east-1",
    config=Config(signature_version=UNSIGNED)
)

bucket = "1000genomes"

prefix = "1000G_2504_high_coverage/working/"

resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix, Delimiter="/")

# Show subdirectories
print("=== SUBFOLDERS ===")
for p in resp.get("CommonPrefixes", []):
    print(p["Prefix"])

# Show files
print("\n=== FILES ===")
for obj in resp.get("Contents", []):
    print(obj["Key"])

=== SUBFOLDERS ===
1000G_2504_high_coverage/working/20201028_3202_phased/
1000G_2504_high_coverage/working/20201028_3202_raw_GT_with_annot/

=== FILES ===


In [5]:
import s3fs
import json

# Create an S3 filesystem object with unsigned access
fs = s3fs.S3FileSystem(anon=True)

s3_file = "s3://1000genomes/1000G_2504_high_coverage/working/20201028_3202_phased/phased-manifest_July2021.tsv"

with fs.open(s3_file, 'r') as f:
    metadata = json.load(f)

# Inspect top-level keys
print(metadata.keys())


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [11]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config

# Create unsigned S3 client
s3 = boto3.client('s3', region_name='us-east-1', config=Config(signature_version=UNSIGNED))

bucket = "1000genomes"
prefix = "1000G_2504_high_coverage/working"

# Get list of VCFs and TBIs
print(f"Listing files under s3://{bucket}/{prefix} ...\n")
paginator = s3.get_paginator('list_objects_v2')
vcf_info = []

for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        size = obj["Size"]
        if key.endswith(".vcf.gz") or key.endswith(".vcf.gz.tbi"):
            vcf_info.append((key, size))

# Print summary
for key, size in sorted(vcf_info):
    flag = "✅ OK" if size > 0 else "❌ EMPTY"
    print(f"{key:<100} {size/1e6:8.2f} MB   {flag}")

# Optional: Check if every VCF has a corresponding .tbi file
print("\n=== Index pairing check ===")
vcf_bases = {k.replace(".tbi", "") for k, _ in vcf_info}
for base in vcf_bases:
    has_vcf = any(k == base for k, _ in vcf_info)
    has_tbi = any(k == base + ".tbi" for k, _ in vcf_info)
    if has_vcf and not has_tbi:
        print(f"⚠️ Missing index for: {base}")


Listing files under s3://1000genomes/1000G_2504_high_coverage/working ...

1000G_2504_high_coverage/working/20201028_3202_phased/CCDG_14151_B01_GRM_WGS_2020-08-05_chr1.filtered.shapeit2-duohmm-phased.vcf.gz     0.00 MB   ❌ EMPTY
1000G_2504_high_coverage/working/20201028_3202_phased/CCDG_14151_B01_GRM_WGS_2020-08-05_chr1.filtered.shapeit2-duohmm-phased.vcf.gz.tbi     0.00 MB   ❌ EMPTY
1000G_2504_high_coverage/working/20201028_3202_phased/CCDG_14151_B01_GRM_WGS_2020-08-05_chr10.filtered.shapeit2-duohmm-phased.vcf.gz     0.00 MB   ❌ EMPTY
1000G_2504_high_coverage/working/20201028_3202_phased/CCDG_14151_B01_GRM_WGS_2020-08-05_chr10.filtered.shapeit2-duohmm-phased.vcf.gz.tbi     0.00 MB   ❌ EMPTY
1000G_2504_high_coverage/working/20201028_3202_phased/CCDG_14151_B01_GRM_WGS_2020-08-05_chr11.filtered.shapeit2-duohmm-phased.vcf.gz     0.00 MB   ❌ EMPTY
1000G_2504_high_coverage/working/20201028_3202_phased/CCDG_14151_B01_GRM_WGS_2020-08-05_chr11.filtered.shapeit2-duohmm-phased.vcf.gz.tbi     0.0

In [4]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config

# Unsigned S3 client (for public buckets like 1000 Genomes)
s3 = boto3.client('s3', region_name='us-east-1', config=Config(signature_version=UNSIGNED))

bucket = "1000genomes"
prefix = "release/20130502/ALL.chr21"

# Use paginator to list all files
paginator = s3.get_paginator('list_objects_v2')
vcf_info = []

for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get('Contents', []):
        key = obj['Key']
        if key.endswith(".vcf.gz.tbi"):
            # Check if the index exists by looking for a .tbi file in the same prefix
            index_key = key + ".tbi"
            try:
                s3.head_object(Bucket=bucket, Key=index_key)
                has_index = True
            except s3.exceptions.ClientError:
                has_index = False

            vcf_info.append({
                "vcf": key,
                "size_MB": obj['Size'] / 1024**2,
                "has_index": has_index
            })

# Print results
for info in vcf_info:
    print(f"{info['vcf']}  {info['size_MB']:.2f} MB  Index: {info['has_index']}")


release/20130502/ALL.chr21.phase3_shapeit2_mvncall_integrated_v5a.20130502.genotypes.vcf.gz.tbi  0.03 MB  Index: False


In [44]:
# === Explore 1000 Genomes AWS folders (clean navigation) ===

import boto3
from botocore import UNSIGNED
from botocore.config import Config
import pandas as pd

pd.set_option("display.max_colwidth", None)

# public S3 client
s3 = boto3.client(
    "s3",
    region_name="us-east-1",
    config=Config(signature_version=UNSIGNED)
)

BUCKET = "1000genomes"

PREFIXES = {
    "2504_high_coverage": "1000G_2504_high_coverage/",
    "additional_698_related": "1000G_2504_high_coverage/additional_698_related/",
}

# === Simple file lister ===

def list_s3_files(bucket, prefix):
    rows = []
    paginator = s3.get_paginator("list_objects_v2")

    for page in paginator.paginate(
        Bucket=bucket,
        Prefix=prefix
    ):
        for obj in page.get("Contents", []):
            rows.append({
                "file": obj["Key"],
                "size_gb": obj["Size"] / 1e9,
                "s3_uri": f"s3://{bucket}/{obj['Key']}"
            })

    return pd.DataFrame(rows)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
list_s3_files(BUCKET, PREFIXES["additional_698_related"])

,file,size_gb,s3_uri
0,1000G_2504_high_coverage/additional_698_related/1000G_698_related_high_coverage.sequence.index,0.000293,s3://1000genomes/1000G_2504_high_coverage/additional_698_related/1000G_698_related_high_coverage.sequence.index
1,1000G_2504_high_coverage/additional_698_related/20130606_g1k_3202_samples_ped_population.txt,0.000097,s3://1000genomes/1000G_2504_high_coverage/additional_698_related/20130606_g1k_3202_samples_ped_population.txt
2,1000G_2504_high_coverage/additional_698_related/20200526_1000G_2504plus698_high_cov_data_reuse_README.txt,0.000002,s3://1000genomes/1000G_2504_high_coverage/additional_698_related/20200526_1000G_2504plus698_high_cov_data_reuse_README.txt
3,1000G_2504_high_coverage/additional_698_related/data/ERR3988761/HG00405.final.cram,15.158156,s3://1000genomes/1000G_2504_high_coverage/additional_698_related/data/ERR3988761/HG00405.final.cram
4,1000G_2504_high_coverage/additional_698_related/data/ERR3988761/HG00405.final.cram.crai,0.001334,s3://1000genomes/1000G_2504_high_coverage/additional_698_related/data/ERR3988761/HG00405.final.cram.crai
5,1000G_2504_high_coverage/additional_698_related/data/ERR3988762/HG00408.final.cram,15.021937,s3://1000genomes/1000G_2504_high_coverage/additional_698_related/data/ERR3988762/HG00408.final.cram
6,1000G_2504_high_coverage/additional_698_related/data/ERR3988762/HG00408.final.cram.crai,0.001378,s3://1000genomes/1000G_2504_high_coverage/additional_698_related/data/ERR3988762/HG00408.final.cram.crai
7,1000G_2504_high_coverage/additional_698_related/data/ERR3988763/HG00418.final.cram,16.325548,s3://1000genomes/1000G_2504_high_coverage/additional_698_related/data/ERR3988763/HG00418.final.cram
8,1000G_2504_high_coverage/additional_698_related/data/ERR3988763/HG00418.final.cram.crai,0.001392,s3://1000genomes/1000G_2504_high_coverage/additional_698_related/data/ERR3988763/HG00418.final.cram.crai
9,1000G_2504_high_coverage/additional_698_related/data/ERR3988764/HG00420.final.cram,14.954039,s3://1000genomes/1000G_2504_high_coverage/additional_698_related/data/ERR3988764/HG00420.final.cram
